# Chromatin Feature Generation Vignette 

### To begin this notebook, fragment files and celltype annotations are required 
### The notebook contains steps to process fragments, call peaks, merge peaks, and generate matrices from snATAC-seq, snK27me3, and snK27ac  

In [2]:
import os 
import subprocess

import numpy as np
import pandas as pd 

In [3]:
wd = os.getcwd()
print(f"Current Working Directory: {wd}")

Current Working Directory: /tscc/projects/ps-epigen/bpToolbox/Callpeak_to_Matrix


In [4]:
os.makedirs(f'{wd}/logs/', exist_ok = True)

## Steps:

### 01. Splitfrags 

In [27]:
### 01. Split Frags 

### Inputs 
### Provide inputs to this notebook to configure kicker script 


### ********* input 1 **********
### tscc user name 
user = 'rlancione'

### ********* input 2 **********
### I2. frag paths 
frag_paths = [
    "/tscc/projects/ps-epigen/10x_output/240813/counts_out/Multiome/QY_2579_QY_2578/outs/atac_fragments.tsv.gz",
    "/tscc/projects/ps-epigen/10x_output/240813/counts_out/Multiome/QY_2585_QY_2584/outs/atac_fragments.tsv.gz",
  
]

### ********* input 3 **********
### I3. Celltype column
celltype_col = 'celltype'

### ********* input 4 **********
### I4. Metadata with sample & barcode columns 
metadata = "/tscc/projects/ps-epigen/users/rlan/LFNIH_Pt2/Subclustering/SCVI/multi_metadata.csv"

### ********* input 5 **********
### resources: 2-40 cores depending on size. 
### 10G per core is typically safe unleff frags are extra large 
cores="3"
memory="30"

### ********* input 6 **********
### I6. Output dir 
split_out = '/tscc/lustre/ddn/scratch/rlancione/splitfrags_out/'

In [6]:
### write frags to file 
pd.DataFrame.from_dict({'frag_paths': frag_paths}).to_csv('frags.txt')

In [7]:
### multiply samples by groups to estimate combinations and cores 

In [18]:
### Configure split frags kicker 
with open(f"{wd}/shell_templates/splitfrags_kick.sh", 'rt') as fh:
    data = fh.read()
    #replace all occurrences of the required string
    data = data.replace('[USR]', user)
    data = data.replace('[CORES]', cores)
    data = data.replace('[MEM]', memory)
    data = data.replace('[WDIR]', f'{wd}/')
    data = data.replace('[CTYPE_COL]', celltype_col)
    data = data.replace('[META_PATH]', metadata)
    data = data.replace('[OUTDIR]', split_out)


    
with open(f"{wd}/shell/splitfrags_kick.sh", "w") as fh:
    fh.write(data)

In [19]:
### Launch split frags 
sbatch_cmd = f"sbatch {wd}/shell/splitfrags_kick.sh"
print(sbatch_cmd)
subprocess.run(sbatch_cmd, shell = True)

sbatch /tscc/projects/ps-epigen/bpToolbox/Callpeak_to_Matrix/shell/splitfrags_kick.sh
Submitted batch job 9155542


CompletedProcess(args='sbatch /tscc/projects/ps-epigen/bpToolbox/Callpeak_to_Matrix/shell/splitfrags_kick.sh', returncode=0)

In [ ]:
### This step may take some time to complete. 
### Output fragments are required for the next step: Calling Peaks 

In [23]:
# Check for outputs before proceeding 
# Could add sizes. But looks good to me 
os.listdir(f'{split_out}/merged')

['NK.bed',
 'HSC.bed',
 'Hepatocytes.bed',
 'Myeloid.bed',
 'Cholangiocyte.bed',
 'Schwann.bed',
 'Mast.bed',
 'T.bed',
 'Endothelial.bed',
 'B.bed']

### 02. Peakcalling 

In [2]:
import sys
sys.path.append("/tscc/projects/ps-epigen/bpToolbox/snATAC/")
import macs3
#dir(macs3)

In [7]:
### Inputs 

### 01. Output dir 
callpeak_out='/tscc/lustre/ddn/scratch/rlancione/macs3_out/'
os.makedirs(callpeak_out, exist_ok=True)

### 02. Assay 
assay='atac'

In [13]:
### Check input files 
ipath=f'{split_out}merged/'
files=os.listdir(ipath)
files

['NK.bed',
 'HSC.bed',
 'Hepatocytes.bed',
 'Myeloid.bed',
 'Cholangiocyte.bed',
 'Schwann.bed',
 'Mast.bed',
 'T.bed',
 'Endothelial.bed',
 'B.bed']

In [14]:
### Call peaks on each one 
for f in files:
    macs3.callpeak(ipath=f'{ipath}{f}', opath=callpeak_out, assay=assay)

In [15]:
os.listdir(callpeak_out)

['Schwann_summits.bed',
 'HSC_peaks.narrowPeak',
 'Endothelial_summits.bed',
 'T_peaks.xls',
 'HSC_summits.bed',
 'Mast_log.txt',
 'Hepatocytes_peaks.xls',
 'Mast_peaks.narrowPeak',
 'Schwann_log.txt',
 'Cholangiocyte_summits.bed',
 'HSC_log.txt',
 'B_log.txt',
 'B_peaks.xls',
 'Hepatocytes_peaks.narrowPeak',
 'T_summits.bed',
 'Myeloid_peaks.narrowPeak',
 'NK_log.txt',
 'Myeloid_summits.bed',
 'NK_peaks.xls',
 'Mast_peaks.xls',
 'Endothelial_peaks.xls',
 'T_log.txt',
 'Hepatocytes_summits.bed',
 'Schwann_peaks.xls',
 'NK_peaks.narrowPeak',
 'HSC_peaks.xls',
 'B_summits.bed',
 'NK_summits.bed',
 'T_peaks.narrowPeak',
 'B_peaks.narrowPeak',
 'Hepatocytes_log.txt',
 'Myeloid_log.txt',
 'Cholangiocyte_log.txt',
 'Cholangiocyte_peaks.narrowPeak',
 'Endothelial_peaks.narrowPeak',
 'Mast_summits.bed',
 'Cholangiocyte_peaks.xls',
 'Myeloid_peaks.xls',
 'Endothelial_log.txt',
 'Schwann_peaks.narrowPeak']

### 03. Peak Merging

In [ ]:
### Here the method of peak merging also depends on the assay
### Iteritive merging for atac, but simpler peak reductions for pairtag 

In [19]:
### Inputs 

### 01. Peakmerge output dir
mergepeak_out = '/tscc/lustre/ddn/scratch/rlancione/mergepeak_out/'
os.makedirs(mergepeak_out, exist_ok=True)

### 02. Name 
# e.g me3_unionBroad
name='multiome_unionNarrow'

### organism
### For atac it determines chromsizes and blacklist 
### for now only hg38

#### Use either the pairtag or atac merge strategy

In [ ]:
#### ******* Pairtag merging *******
if assay == 'k27me3 or k27ac':
    
    ### Configure bedtools merge.
    with open(f"{wd}/shell_templates/pairtag_peakmerge.sh", 'rt') as fh:
        data = fh.read()
        #replace all occurrences of the required string
        data = data.replace('[IPEAKS]', callpeak_out)
        data = data.replace('[ODIR]', peakmerge_out)
        data = data.replace('[NAME]', name)

    with open(f"{wd}/shell/pairtag_merge.sh", "w") as fh:
        fh.write(data)

    ### Launch split frags 
    sbatch_cmd = f"./{wd}/shell/pairtag_merge.sh"
    print(sbatch_cmd)
    subprocess.run(sbatch_cmd, shell = True)

In [20]:
#### ******* ATAC merging ********
if assay == 'atac':

    ### configure iterative peakmerging 
    files = os.listdir(callpeak_out)
    summit_files = list(filter(lambda file:'summits.bed' in file, files))
    summit_paths = [callpeak_out + sf for sf in summit_files]
    
    celltypes = [sf.replace("_summits.bed","") for sf in summit_files]

    data = {'Celltype': celltypes,
            'Paths': summit_paths}
    
    df = pd.DataFrame(data)
    df.to_csv('my_summits.txt', sep="\t", index=False, header=False)

    ### Configure bedtools merge.
    with open(f"{wd}/shell_templates/iter_peakmerge.sh", 'rt') as fh:
        data = fh.read()
        #replace all occurrences of the required string
        data = data.replace('[OUTDIR]', mergepeak_out)
        data = data.replace('[NAME]', name)

    with open(f"{wd}/shell/iter_peakmerge.sh", "w") as fh:
        fh.write(data)

    ### Launch split frags 
    sbatch_cmd = f"source {wd}/shell/iter_peakmerge.sh"
    print(sbatch_cmd)
    subprocess.run(sbatch_cmd, shell = True)



source /tscc/projects/ps-epigen/bpToolbox/Callpeak_to_Matrix/shell/iter_peakmerge.sh


Warning messages:
1: package ‘GenomicRanges’ was built under R version 4.3.3 
2: package ‘BiocGenerics’ was built under R version 4.3.2 
3: package ‘S4Vectors’ was built under R version 4.3.3 
4: package ‘IRanges’ was built under R version 4.3.3 
Warning messages:
1: package ‘Biostrings’ was built under R version 4.3.3 
2: package ‘XVector’ was built under R version 4.3.3 
3: package ‘BiocIO’ was built under R version 4.3.2 
4: package ‘rtracklayer’ was built under R version 4.3.3 

Attaching package: ‘tictoc’

The following object is masked from ‘package:GenomicRanges’:

    shift

The following object is masked from ‘package:IRanges’:

    shift

The following object is masked from ‘package:data.table’:

    shift

working on summit set for... Endothelial
Converged after 3 iterations!
working on summit set for... HSC
Converged after 2 iterations!
working on summit set for... Cholangiocyte
Converged after 3 iterations!
working on summit set for... T
Converged after 2 iterations!
worki

parse summit peak set: 10.024 sec elapsed
write filtered & fixed peak set: 0.175 sec elapsed


Warning messages:
1: In .merge_two_Seqinfo_objects(x, y) :
  Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': GL000009.2
  - in 'y': KI270721.1, KI270731.1
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning).
2: In .merge_two_Seqinfo_objects(x, y) :
  Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': GL000009.2, KI270721.1
  - in 'y': KI270711.1, KI270734.1
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning).
3: In .merge_two_Seqinfo_objects(x, y) :
  Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': GL000009.2, GL000218.1, KI270727.1, KI270721.1, KI270731.1, KI270711.1, KI270734.1
  - in 'y': KI270726.1
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning)

merge to union peak list: 0.576 sec elapsed
save union peak set: 0.06 sec elapsed


Converged after 4 iterations!


In [21]:
os.listdir(mergepeak_out)

['HSC.filterNfixed.peakset',
 'Cholangiocyte.filterNfixed.peakset',
 'NK.filterNfixed.peakset',
 'T.filterNfixed.peakset',
 'Hepatocytes.filterNfixed.peakset',
 'B.filterNfixed.peakset',
 'multiome_unionNarrow.filteredNfixed.union.peakSet',
 'Myeloid.filterNfixed.peakset',
 'Endothelial.filterNfixed.peakset']



### 04. Matrix Generation

In [42]:
import episcanpy as epsc

ModuleNotFoundError: No module named 'episcanpy'

In [ ]:
### argparse for support 
### TEMPLATE for split_args 

In [ ]:
### rest of notebook can run on notebook resources??

In [32]:
### Matrix generation inputs 

### 01. Regions. Either union peakset or other 
regions = f'{mergepeak_out}multiome_unionNarrow.filteredNfixed.union.peakSet'

### 02. Anndata/Matrix Generation Outdir 
adata_out = '/tscc/lustre/ddn/scratch/rlancione/atac_adatas_out/'

In [37]:
metadata

'/tscc/projects/ps-epigen/users/rlan/LFNIH_Pt2/Subclustering/SCVI/multi_metadata.csv'

In [40]:
def build_obj_from_peaks(sample, frag, metapath, regions, outdir):

    metadata = pd.read_csv(metapath, index_col=0)
    metadata['sample'] = ["_".join(i.split("_")[:-1]) for i in metadata.index]
    metadata['barcode'] = [i.split("_")[-1] for i in metadata.index]
    tmeta = metadata[metadata['sample'] == sample]
    barcodes = tmeta['barcode']

    ### Dev version fixed it ****** 
    adata = epsc.count_matrix.peak_mtx(frag, regions, barcodes)
    
    adata.obs.index = [f'{sample}_{b}' for b in adata.obs.index]

    adata.write_h5ad(f'{outdir}{sample}.h5ad')

In [29]:
samples = [f.split('/')[-3] for f in frag_paths]
samples

['QY_2579_QY_2578', 'QY_2585_QY_2584']

In [41]:
for sample, frag in samples, frag_paths:
    print(f'Building {sample}')
    build_obj_from_peaks(sample, frag, metadata, regions, adata_out)

Building QY_2579_QY_2578


/scratch/rlancione/job_9163547/ipykernel_652954/3903542304.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(metapath, index_col=0)


NameError: name 'epsc' is not defined